E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [7]:
import torch
import torch.nn.functional as F

In [8]:
from torch.utils.data.dataset import random_split

In [9]:
# open file
words = open('../names.txt', 'r').read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [10]:
# 拆分训练集

def SplitSet(words):
    g = torch.Generator().manual_seed(22)
    n = len(words)
    c_train = int(n * 0.8)
    c_dev = int(n * 0.1)
    c_test = n - c_train - c_dev
    split = random_split(words, [c_train, c_dev, c_test], generator=g)
    return [list(split[0]), list(split[1]), list(split[2])]

n_train, n_dev, n_test = SplitSet(words)

Bigram

In [80]:
class Bigram:

    def __init__(self):
        g = torch.Generator().manual_seed(22)
        self.W = torch.randn((27, 27), generator=g, requires_grad=True)

    def initWrods(self, inWords = []):
        chars = sorted(list(set(''.join(inWords))))
        self.stoi = {s:i+1 for i,s in enumerate(chars) }
        self.stoi['.'] = 0
        self.itos = {i:s for s,i in self.stoi.items()}

        xs, self.ys = [], []
        for w in inWords:
            chs = ['.'] + list(w) + ['.']
            for ch1, ch2 in zip(chs, chs[1:]):
                ix1 = self.stoi[ch1]
                ix2 = self.stoi[ch2]
                xs.append(ix1)
                self.ys.append(ix2)
        xs = torch.tensor(xs)
        self. ys = torch.tensor(self.ys)
        self.num = xs.nelement()
        self.xenc = F.one_hot(xs, num_classes=27).float()

    def getLoss(self):
        # xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
        logits = self.xenc @ self.W # predict log-counts
        counts = logits.exp() # counts, equivalent to N
        probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
        loss = -probs[torch.arange(self.num), self.ys].log().mean()
        return loss
        
    def train(self, inTrainwords):
        self.initWrods(inTrainwords)
        for i in range(500):
            loss = self.getLoss()
            if (i % 40 == 0 or i == 499):
                print(f'loss = {loss}')
            # backward pass
            self.W.grad = None # set to zero the gradient
            loss.backward()
            # update
            self.W.data += -70 * self.W.grad
        
        
    def output(self):
        g = torch.Generator().manual_seed(22)
        for i in range(5):
            out = []
            ix = 0
            while True:
                xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
                logits = xenc @ self.W # predict log-counts
                counts = logits.exp() # counts, equivalent to N
                p = counts / counts.sum(1, keepdims=True) # probabilities for next character
                ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
                out.append(self.itos[ix])
                if ix == 0:
                    break
        print(''.join(out))

    def test(self, intestWords):
        self.initWrods(intestWords)
        loss = self.getLoss()
        print(f'test loss = {loss}')        

In [81]:
bi = Bigram()

In [82]:
bi.train(n_train)

loss = 3.7930970191955566
loss = 2.490963935852051
loss = 2.4701457023620605
loss = 2.4635701179504395
loss = 2.4605863094329834
loss = 2.4588921070098877
loss = 2.4577949047088623
loss = 2.457026481628418
loss = 2.456460475921631
loss = 2.456028461456299
loss = 2.455688238143921
loss = 2.4554154872894287
loss = 2.4551913738250732
loss = 2.455098867416382


In [83]:
bi.test(n_test)
bi.test(n_dev)

test loss = 2.4641757011413574
test loss = 2.4578969478607178


Trigram

In [115]:
t = torch.tensor([(1,2),(3,4)])
tenc = F.one_hot(t)
print(tenc)
qq = tenc.view(-1,2,2)
qq.nelement()

tensor([[[0, 1, 0, 0, 0],
         [0, 0, 1, 0, 0]],

        [[0, 0, 0, 1, 0],
         [0, 0, 0, 0, 1]]])


20

In [164]:
class Trigram:
    def __init__(self, smoothnesses):
        g = torch.Generator().manual_seed(22)
        # 两个字符预测一个输出 如果用27**27则是把2个字符视作一个输入，排列组合后作为输入
        self.W = torch.randn((27 * 2, 27), generator = g, requires_grad = True) 
        self.initIndex()
        self.smoothnesses = smoothnesses

    def initIndex(self):
        chars = '.abcdefghijklmnopqrstuvwxyz'
        self.stoi = {s:i for i,s in enumerate(chars)}
        self.itos = {i:s for i,s in enumerate(chars)}

    def initWords(self, inWords):
        xs, self.ys = [], []
        for w in inWords:
            chs = ['.'] + ['.'] + list(w) + ['.']
            for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
                    ix1 = self.stoi[ch1]
                    ix2 = self.stoi[ch2]
                    ix3 = self.stoi[ch3]
                    xs.append((ix1, ix2))
                    self.ys.append(ix3)
                    # print(f'{firstTwo}, {ch}')
        xs = torch.tensor(xs)
        self.ys = torch.tensor(self.ys)
        self.num = self.ys.nelement()
        self.xenc = F.one_hot(xs, num_classes = 27).float()
        # print(self.xenc.shape)

    def getLoss(self):
        logits = self.xenc.view(-1, 27*2) @ self.W # view 把xenc从 X*27 转为 x/2 * 27 * 2 
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims = True)
        
        #loss
        loss = -probs[torch.arange(self.num), self.ys].log().mean() + self.smoothnesses * (self.W**2).mean()
        return loss

    def train(self, inWords):
        self.initWords(inWords)

        for i in range(2000):
            loss = self.getLoss()
            if (i % 100 == 0 or i == 99):
                print(f'loss: {loss}')
            self.W.grad = None
            loss.backward()

            self.W.data += -20 * self.W.grad

    def evaluate(self, inWords):
        xs = []
        lossi = []
        for w in inWords:
            chs = ['.'] + ['.'] + list(w) + ['.']
            for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
                    ix1 = self.stoi[ch1]
                    ix2 = self.stoi[ch2]
                    ix3 = self.stoi[ch3]

                    xenc = F.one_hot(torch.tensor((ix1, ix2)), num_classes = 27).float()
                    logits = xenc.view(-1,27*2) @ self.W
                    counts = logits.exp()
                    probs = counts / counts.sum(1, keepdims = True)
                    # print(probs.shape)
                    lossi.append(probs[0, ix3].log())
        print(-torch.tensor(lossi).mean().item())


    def test(self, inWords):
        self.initWords(inWords)

        loss = self.getLoss()
        print(f'loss: {loss}')

    def output(self):
        g = torch.Generator().manual_seed(22)
        for i in range(0,3):
            out = '..'
            ix = 0
            while True:
                outxenc = F.one_hot(torch.tensor([ix]), num_classes = 729).float()
                logits = outxenc @ self.W
                counts = logits.exp()
                probs = counts / counts.sum(1, keepdims = True)
                i = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
                out += self.itc[i]
                ix = self.sti[out[-2:]]
                # print(out[-2:])
                if (i == 0):
                    break
            print(f'{out}')

In [165]:
tri = Trigram(0.0)

In [166]:
tri.train(n_train)

loss: 4.066823482513428
loss: 2.404068946838379
loss: 2.4033823013305664
loss: 2.3682329654693604
loss: 2.3561758995056152
loss: 2.3502180576324463
loss: 2.346712589263916
loss: 2.3444364070892334
loss: 2.342862844467163
loss: 2.3417251110076904
loss: 2.3408713340759277
loss: 2.3402116298675537
loss: 2.339688301086426
loss: 2.339263677597046
loss: 2.3389129638671875
loss: 2.338618040084839
loss: 2.338366985321045
loss: 2.3381502628326416
loss: 2.337961196899414
loss: 2.337794780731201
loss: 2.3376471996307373


In [167]:
tri.evaluate(n_test)
tri.evaluate(n_dev)

2.3477444648742676
2.338024377822876


E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

In [ ]:
tri.output()

In [168]:
triSoft1 = Trigram(0.1)

In [169]:
triSoft1.train(n_train)

loss: 4.163815498352051
loss: 2.4605534076690674
loss: 2.4599030017852783
loss: 2.4291319847106934
loss: 2.4209823608398438
loss: 2.418022632598877
loss: 2.416785955429077
loss: 2.4162261486053467
loss: 2.4159586429595947
loss: 2.415825843811035
loss: 2.4157583713531494
loss: 2.4157235622406006
loss: 2.4157049655914307
loss: 2.4156949520111084
loss: 2.415689706802368
loss: 2.41568660736084
loss: 2.415684938430786
loss: 2.4156839847564697
loss: 2.4156835079193115
loss: 2.4156832695007324
loss: 2.4156830310821533


In [171]:
triSoft1.evaluate(n_dev)
triSoft1.evaluate(n_train)

2.362800359725952
2.3636603355407715
